In [ ]:
%load_ext autoreload
%autoreload 2

This notebook determines the range of fire areas in which the 3 fire area datasets are in agreement by minimizing the Kuiper statistic of their CDFs over the region where the datasets overlap spatially and temporally.

In [ ]:
import numpy as np

from wildfire_stats import config as cfg
from wildfire_stats.data import io as dio
from wildfire_stats.plotting import dataset_comp as pdc
from wildfire_stats.reporting import summaries as rsum

P = cfg.load_paths()
A = cfg.load_assumptions()

### MTBS vs. GFD

In [ ]:
mtbs_proj = dio.load_mtbs(P.mtbs / "S_USA.MTBS_BURN_AREA_BOUNDARY.shp", area_col="area_km2")
idaho_gdf = dio.load_idaho(P.idaho / "HFD_HistoricFiresDatabase7_0.gdb")

# MTBS starts in 1984; GFA / MODIS cover modis_years
mtbs_proj = dio.filter_years(mtbs_proj, "YEAR", (1984, 2016))
idaho_gdf = dio.filter_years(idaho_gdf, "FIRE_YEAR", (1984, 2016))
idaho_gdf2 = dio.filter_years(idaho_gdf, "FIRE_YEAR", A["modis_years"])

mtbs_overlap = dio.overlap_with(mtbs_proj, idaho_gdf, dedup_col="FIRE_ID")

rsum.print_summary("MTBS", mtbs_overlap, "YEAR")
rsum.print_summary("Idaho", idaho_gdf, "FIRE_YEAR")

In [ ]:
alphas = np.linspace(0, 300, 2000)

pdc.plot_cdf_vs_area(mtbs_overlap, idaho_gdf, labels=("MTBS", "Idaho"))
pdc.plot_alpha_vs_kuiper_with_min_cdf(mtbs_overlap, idaho_gdf, alphas, labels=("MTBS", "Idaho"))

### GFA vs. GFD

In [ ]:
gfa_combined_gdf = dio.load_gfa(P.gfa)

gfa_overlap = dio.overlap_with(gfa_combined_gdf, idaho_gdf2, dedup_col="geometry")

In [ ]:
rsum.print_summary("GFA (overlap)", gfa_overlap, "YEAR")
rsum.print_summary("Idaho", idaho_gdf2, "FIRE_YEAR")

In [ ]:
alphas2 = np.linspace(0, 200, 2000)

pdc.plot_cdf_vs_area(gfa_overlap, idaho_gdf2, labels=("GFA", "Idaho"))
pdc.plot_alpha_vs_kuiper_with_min_cdf(gfa_overlap, idaho_gdf2, alphas2, labels=("GFA", "Idaho"))

### GFA vs. MTBS

In [ ]:
pdc.plot_cdf_vs_area(gfa_overlap, mtbs_overlap, labels=("GFA", "MTBS"))
pdc.plot_alpha_vs_kuiper_with_min_cdf(gfa_overlap, mtbs_overlap, alphas2, labels=("GFA", "MTBS"))

Based on the minimum fire size cutoffs that minimize the Kuiper statistic, all future notebooks will remove fires less than 4 km^2.